# Regulatory Auto-Update — Dev Log

## Objetivo e papel no pipeline

`core/regulatory_auto_update` detecta mudanças no corpus local de
`regulatory_rag` via manifesto de hashes SHA-256 por arquivo — o
pré-requisito técnico para uma futura atualização (manual ou
semi-automatizada) do corpus regulatório, sabendo exatamente o que mudou
antes de reindexar.

**Escopo honesto**: não busca o texto oficial da LGPD em nenhuma fonte
externa — isso exigiria validação jurídica humana antes de qualquer
substituição automática de conteúdo legal, fora de escopo desta versão.

In [1]:
import sys
from pathlib import Path

REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

import tempfile
from pathlib import Path
from core.regulatory_auto_update.manifest import build_manifest, diff_against_manifest, save_manifest

demo_dir = Path(tempfile.mkdtemp(prefix="auto_update_demo_"))
manifest_path = demo_dir / "manifest.json"

manifest = build_manifest()
save_manifest(manifest, manifest_path=manifest_path)
print(f"Manifesto salvo: {len(manifest.files)} arquivo(s), gerado em {manifest.generated_at.isoformat()}")

diff = diff_against_manifest(manifest)
print("Diff contra o corpus atual (sem mudanças):", diff.summary)

# Simula uma "mudança" de forma NÃO destrutiva: usa um manifesto com um hash
# propositalmente desatualizado, sem tocar o corpus real em disco.
fake_previous = manifest.model_copy(deep=True)
fake_previous.files[0] = fake_previous.files[0].model_copy(update={"content_hash": "0" * 64})
diff2 = diff_against_manifest(fake_previous)
print(f"Diff contra manifesto desatualizado (simulado): {diff2.summary}")
print("Modificado (detectado):", diff2.modified)

Manifesto salvo: 12 arquivo(s), gerado em 2026-08-20T23:50:58.245993+00:00
Diff contra o corpus atual (sem mudanças): Nenhuma mudança detectada no corpus (12 arquivo(s) inalterado(s)).
Diff contra manifesto desatualizado (simulado): Mudanças detectadas no corpus: 1 modificado(s). 11 inalterado(s).
Modificado (detectado): ['art_11_bases_legais_sensivel.txt']


## Rodando a suíte de testes

```
"C:/Users/Yuri_/.venvs/athenagov-ai/Scripts/python.exe" -m pytest core/regulatory_auto_update/tests -v
```

9 testes: manifesto cobre os 12 artigos reais; determinístico; roundtrip
save/load; diff sem mudanças; diff detecta arquivo modificado/adicionado/
removido — os 3 últimos testes copiam o corpus real para um diretório
temporário (`shutil.copytree`) e editam só a cópia, nunca o corpus real do
projeto.

## Handoff Summary

- **Status:** ✅ done — 9/9 testes passando.
- **TODO onda futura (fora de escopo desta versão, deliberadamente):**
  qualquer integração com fonte oficial externa de texto legal exige
  validação jurídica humana antes de automatizar — não é uma limitação
  técnica, é uma decisão de risco.